# Summer heatwave footprints around Milan

This case follows maximum daily 2 m air temperature from a single grid cell to the total affected area and the daily footprint map. Use outputs from a verified ERA5-Land run for summer 2022. The source attribution, dates and coverage displayed below come from the selected run.

Run the pipeline first. Place `target.nc`, `events.nc`, `footprints.nc` and `footprints.csv` together under `runs/milan`, or set `HEATWAVE_RUN_DIR` to the run directory before launching Jupyter. Relative paths are resolved from the repository root.

In [ ]:
import os
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import xarray as xr

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
run = root / os.environ.get("HEATWAVE_RUN_DIR", "runs/milan")
required = ["target.nc", "events.nc", "footprints.nc", "footprints.csv"]
missing = [name for name in required if not (run / name).is_file()]
if missing:
    raise FileNotFoundError(f"Complete the real-data pipeline in {run.name}. Missing {', '.join(missing)}")

target = xr.load_dataset(run / "target.nc")
events = xr.load_dataset(run / "events.nc")
labels = xr.load_dataset(run / "footprints.nc")
summary = pd.read_csv(run / "footprints.csv", parse_dates=["date"])
source = str(target.attrs.get("source", ""))
source_key = source.lower().replace("_", "-").replace("‑", "-")
if ("era5-land" not in source_key or "synthetic" in source_key or "fixture" in source_key):
    raise ValueError(f"Expected normalized real ERA5-Land input, found source {source!r}")
for cube in (target, events, labels):
    if cube.attrs.get("contract_version") != "1":
        raise ValueError("The Milan notebook expects file contract v1.")
    np.testing.assert_allclose(cube.lat.values, np.arange(451, 460) / 10, rtol=0, atol=1e-5)
    np.testing.assert_allclose(cube.lon.values, np.arange(86, 101) / 10, rtol=0, atol=1e-5)

years = [int(value) for value in str(events.attrs["baseline_years"]).split(",")]
if years != list(range(1991, 2021)) or target.attrs.get("day_timezone") != "UTC":
    raise ValueError("Expected a 1991–2020 reference period and UTC daily maxima.")
np.testing.assert_array_equal(target.time.values, pd.date_range("2022-05-01", "2022-09-30").values)
days = pd.date_range("2022-06-01", "2022-08-31")
np.testing.assert_array_equal(events.time.values, days.values)
xr.testing.assert_equal(events.tx_c, target.tx_c.sel(time=days))
flag, ids = xr.align(events.heatwave_flag, labels.footprint_id, join="exact")
np.testing.assert_array_equal(np.sign(ids), flag)
print(f"Input source: {source}")
print(f"Reference: {years[0]}–{years[-1]}, {events.attrs['window_days']}-day window.")

## Temperature and local threshold

We use the delivered grid centre at 45.5°N, 9.2°E as a reproducible location near central Milan. The red markers identify dates in a qualifying run at that cell. A missing value leaves a gap in its line.

In [ ]:
cell = events.sel(lat=45.5, lon=9.2, method="nearest")
qualifies = cell.heatwave_flag.values == 1
fig, ax = plt.subplots(figsize=(10, 3.6), layout="constrained")
ax.plot(days, cell.tx_c, color="#164c6b", label="Maximum 2 m air temperature")
ax.plot(days, cell.p90_c, "--", color="#ac6a16", label="Local P90 (15-day window)")
ax.scatter(days[qualifies], cell.tx_c.values[qualifies], s=19, color="#ad3d35",
           zorder=3, label="Qualifying heatwave day")
ax.set(title=f"Daily temperature at {cell.lat.item():.1f}°N, {cell.lon.item():.1f}°E",
       ylabel="Temperature (°C)", xlabel="UTC date", xlim=(days[0], days[-1]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)
plt.show()

for name in ("tx_c", "p90_c"):
    values = cell[name].values
    finite = values[np.isfinite(values)]
    print(f"{name}: {finite.min():.2f} to {finite.max():.2f} °C" if finite.size else f"{name}: no observations")

## Observed affected area and coverage

Sum the component areas by day and check against areas computed from the flag raster. A partially observed day can only establish the affected area in known cells. Wholly unknown days remain gaps.

In [ ]:
def cell_edges(centres):
    mid = (centres[1:] + centres[:-1]) / 2
    return np.r_[2 * centres[0] - mid[0], mid, 2 * centres[-1] - mid[-1]]

lat_edges = cell_edges(flag.lat.values.astype(float))
lon_edges = cell_edges(flag.lon.values.astype(float))
radius_km = float(labels.attrs["earth_radius_km"])
cell_km2 = radius_km**2 * np.outer(np.diff(np.sin(np.deg2rad(lat_edges))),
                                    np.diff(np.deg2rad(lon_edges)))
area = ((flag.values == 1) * cell_km2).sum(axis=(1, 2))
if not summary["date"].isin(days).all():
    raise ValueError("Component CSV contains dates outside the event cube.")
csv_area = summary.groupby("date")["area_km2"].sum().reindex(days, fill_value=0)
np.testing.assert_allclose(area, csv_area.to_numpy(dtype=float), rtol=1e-5, atol=1e-6)
coverage = (flag.values != -1).mean(axis=(1, 2))
area[coverage == 0] = np.nan
partial = (coverage > 0) & (coverage < 1)

fig, (ax, cov) = plt.subplots(2, 1, sharex=True, figsize=(10, 5),
                              height_ratios=[3, 1], layout="constrained")
ax.plot(days, area, color="#17614f", label="Observed area")
ax.scatter(days[partial], area[partial], color="#ac6a16", s=20, label="Incomplete coverage", zorder=3)
ax.set(title="Area of qualifying grid cells in the study rectangle", ylabel="Area (km²)")
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)
cov.plot(days, 100 * coverage, color="#36586a")
cov.set(ylabel="Known (%)", xlabel="UTC date", ylim=(-5, 105), yticks=[0, 50, 100],
        xlim=(days[0], days[-1]))
cov.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.show()

## Daily components on the peak date

Use the earliest date with the largest positive, unrounded observed area. Each numbered group consists of edge-connected cells for this UTC date. A group touching the map edge may extend beyond the study grid.

In [ ]:
if not np.any(area > 0):
    print("No qualifying cells on observed dates." if np.any(coverage > 0) else "The whole summer is unknown.")
else:
    peak = int(np.nanargmax(area))
    layer = ids.values[peak]
    components = int(layer.max())
    palette = plt.get_cmap("tab20")
    colours = ["#64696f", "#f1f3f4"] + [palette(i % 20) for i in range(components)]
    cmap = ListedColormap(colours)
    norm = BoundaryNorm(np.arange(-1.5, components + 1.5), cmap.N)
    fig, ax = plt.subplots(figsize=(9, 6), layout="constrained")
    ax.pcolormesh(lon_edges, lat_edges, layer, cmap=cmap, norm=norm, shading="flat")
    for component in range(1, components + 1):
        y, x = np.argwhere(layer == component)[0]
        ax.text(flag.lon.values[x], flag.lat.values[y], str(component), ha="center", va="center",
                fontsize=8, bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1))
    ax.set(xlim=(lon_edges[0], lon_edges[-1]), ylim=(lat_edges[0], lat_edges[-1]),
           xlabel="Longitude (°E)", ylabel="Latitude (°N)",
           aspect=1 / np.cos(np.deg2rad(flag.lat.values.mean())),
           title=f"{days[peak]:%d %B %Y} UTC · {area[peak]:,.1f} km² observed · {coverage[peak]:.1%} known")
    ax.legend(handles=[Patch(color=colours[0], label="Unknown"), Patch(color=colours[1], label="Background"),
                       Patch(color=colours[2], label="Numbered daily components")],
              loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
    fig.suptitle(f"Peak daily footprints · {source}", fontsize=12, wrap=True)
    plt.show()